# Lab 4 — Git-Integrated Code Agent

**Goal:** Extend the code agent to read files from a Git repository, fix them, and commit to a new branch.  
This is the core pattern used by Devin, SWE-bench agents, and auto-PR tools.

**What we build:**
1. Read a buggy file from a repo
2. Run the existing tests — confirm they fail
3. Ask LLM to fix using AST context
4. Run tests — confirm they pass
5. Commit on a new branch with a descriptive message

**Requires:** `pip install gitpython`

In [ ]:
import sys, os, ast, tempfile, shutil
from pathlib import Path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from git_agent import read_ast_summary
from sandbox import run_pytest
from openai import OpenAI

try:
    import git
    print('gitpython available ✓')
except ImportError:
    print('gitpython not installed — run: pip install gitpython')

client = OpenAI()

## 1. Create a demo repo with a bug

In [ ]:
# Create a temporary git repo to demonstrate the pattern
DEMO_DIR = Path(tempfile.mkdtemp(prefix='agent_demo_'))

# Buggy source file
buggy_source = '''
def celsius_to_fahrenheit(c: float) -> float:
    # BUG: wrong formula
    return c * 9 / 5  # Missing the + 32

def fahrenheit_to_celsius(f: float) -> float:
    return (f - 32) * 5 / 9  # This one is correct
'''

test_source = '''
def test_boiling_point():
    assert celsius_to_fahrenheit(100) == 212.0

def test_freezing_point():
    assert celsius_to_fahrenheit(0) == 32.0

def test_body_temp():
    assert celsius_to_fahrenheit(37) == pytest.approx(98.6, rel=1e-3)

def test_reverse():
    assert fahrenheit_to_celsius(212) == pytest.approx(100.0)
'''

(DEMO_DIR / 'temperature.py').write_text(buggy_source)
(DEMO_DIR / 'test_temperature.py').write_text('import pytest\nfrom temperature import *\n\n' + test_source)

# Init git repo
repo = git.Repo.init(DEMO_DIR)
repo.config_writer().set_value('user', 'name', 'Agent Demo').release()
repo.config_writer().set_value('user', 'email', 'agent@demo.com').release()
repo.index.add(['temperature.py', 'test_temperature.py'])
repo.index.commit('Initial commit with buggy code')

print(f'Demo repo created at: {DEMO_DIR}')
print(f'Commits: {list(repo.iter_commits())}')

## 2. Confirm tests fail on the original code

In [ ]:
result = run_pytest(test_source, buggy_source)
print('Initial test run:')
print(result.summary())

## 3. AST-aware fix

In [ ]:
ast_summary = read_ast_summary(buggy_source)
print('AST structure:')
print(ast_summary)

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'Fix the bug. Output only raw Python code. No fences.'},
        {'role': 'user', 'content': f'''Source:\n{buggy_source}\n\nAST:\n{ast_summary}\n\nTests:\n{test_source}\n\nFix celsius_to_fahrenheit only.'''},
    ],
    temperature=0.1,
)
fixed_source = response.choices[0].message.content
print('\nFixed source:')
print(fixed_source)

## 4. Verify fix passes tests

In [ ]:
result = run_pytest(test_source, fixed_source)
print(result.summary())
assert result.success, 'Fix did not pass all tests!'

## 5. Commit the fix on a new branch

In [ ]:
if result.success:
    (DEMO_DIR / 'temperature.py').write_text(fixed_source)
    fix_branch = repo.create_head('agent/fix-celsius-to-fahrenheit', force=True)
    fix_branch.checkout()
    repo.index.add(['temperature.py'])
    commit = repo.index.commit(
        'agent/autofix: fix celsius_to_fahrenheit formula\n\n'
        'Added missing + 32 offset. Tests now pass.\n\n'
        'Fixed-by: code-agent'
    )
    print(f'Committed: {commit.hexsha[:8]} on branch {fix_branch.name}')
    print(f'Log:\n' + repo.git.log('--oneline'))

## 6. Diff to inspect the change

In [ ]:
diff = repo.git.diff('main', 'agent/fix-celsius-to-fahrenheit')
print(diff)

In [ ]:
# Cleanup
shutil.rmtree(DEMO_DIR)
print('Demo repo cleaned up.')

## Exercise
Extend the agent to:
1. Generate a PR description summarising the change (use the diff as context)
2. Add the PR description as a file `PR_DESCRIPTION.md` in the branch before committing

**Bonus:** Make it loop — if the first fix fails tests, ask for another fix before committing.

In [ ]:
# Your code here
